# 03 — Feature Engineering

Every feature is **row-wise**: it depends only on the customer's own values, never on a dataset aggregate. That property is what makes it safe to compute before the split, and it is why the transformer is stateless.

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
warnings.filterwarnings('ignore')

from src.config import load_config
cfg = load_config()

In [2]:
from src.data.loader import make_dataset
from src.features.engineer import ChurnFeatureEngineer, FEATURE_DOCS
import pandas as pd

ds = make_dataset(cfg)
fe = ChurnFeatureEngineer().fit(ds.X_train)
out = fe.transform(ds.X_train)
print(f'{ds.X_train.shape[1]} raw columns -> {out.shape[1]} columns')

19 raw columns -> 32 columns


## Every feature, and why it should help

In [3]:
pd.DataFrame([{'feature': k, 'type': v[0], 'rationale': v[1]}
              for k, v in FEATURE_DOCS.items()])

,feature,type,rationale
0,tenure_band,categorical,Churn is heavily front-loaded: 53.3% in the fi...
1,is_new_customer,binary,"Isolates the 0-6 month window, the single risk..."
2,n_addons,numeric,Count of the six internet add-ons held. Churn ...
3,has_internet,binary,Separates 'declined an add-on' from 'cannot ha...
4,addon_adoption_rate,numeric,n_addons as a share of those available to that...
5,is_month_to_month,binary,Contract is the strongest single predictor (AU...
6,is_autopay,binary,"Electronic check churns at 45.5%, far above th..."
7,realized_arpu,numeric,TotalCharges / tenure: what the customer has a...
8,price_vs_history,numeric,MonthlyCharges / realized_arpu. Above 1 means ...
9,billing_discrepancy,numeric,TotalCharges - (tenure x MonthlyCharges). The ...


## The features are stateless

Fitting on the training set and fitting on the rows themselves give identical output. If a target-encoded feature were added, this check would fail — which is the alarm.

In [4]:
sample = ds.X_test.head(50)
a = ChurnFeatureEngineer().fit(ds.X_train).transform(sample)
b = ChurnFeatureEngineer().fit(sample).transform(sample)
print('identical:', a.equals(b))

identical: True


## The awkward cases

Never-billed customers (`tenure == 0`) would produce division by zero. `price_vs_history` fills with **1.0** — 'no change versus history' — because filling with 0 would read as a total price collapse at exactly the tenure where churn risk peaks.

In [5]:
cols = ['tenure','TotalCharges','realized_arpu','price_vs_history',
        'billing_discrepancy','is_new_customer']
out.loc[out['tenure'] == 0, cols].head()

,tenure,TotalCharges,realized_arpu,price_vs_history,billing_discrepancy,is_new_customer
602,0,0.0,0.0,1.0,0.0,1
859,0,0.0,0.0,1.0,0.0,1
1068,0,0.0,0.0,1.0,0.0,1
2659,0,0.0,0.0,1.0,0.0,1
3061,0,0.0,0.0,1.0,0.0,1


In [6]:
out[['n_addons','addon_adoption_rate','price_vs_history',
     'billing_discrepancy','charges_per_addon']].describe().round(3)

,n_addons,addon_adoption_rate,price_vs_history,billing_discrepancy,charges_per_addon
count,5621.000,5621.000,5621.000,5621.000,5621.000
mean,2.046,0.341,1.002,0.426,25.457
std,1.838,0.306,0.052,67.437,14.088
min,0.000,0.000,0.636,-370.850,8.740
25%,0.000,0.000,0.980,-28.550,17.967
50%,2.000,0.333,1.000,0.000,20.690
75%,3.000,0.500,1.021,29.350,27.375
max,6.000,1.000,1.451,373.250,77.900


## `billing_discrepancy` extracts the residual, not a third copy

The audit found `TotalCharges` is 99.91% explained by `tenure x MonthlyCharges`. So the *residual* is the only genuinely new information in that column.

In [7]:
import numpy as np
expected = out['tenure'] * out['MonthlyCharges']
print('corr(TotalCharges, tenure*Monthly):',
      round(float(np.corrcoef(out['TotalCharges'], expected)[0,1]), 6))
out['billing_discrepancy'].describe().round(2)

corr(TotalCharges, tenure*Monthly): 0.999555


count    5621.00
mean        0.43
std        67.44
min      -370.85
25%       -28.55
50%         0.00
75%        29.35
max       373.25
Name: billing_discrepancy, dtype: float64

## The encoded matrix

Preprocessing lives inside the pipeline, so scaler statistics and encoder categories are fitted on training folds only.

In [8]:
from sklearn.linear_model import LogisticRegression
from src.features.preprocess import build_pipeline, output_feature_names

pipe = build_pipeline(LogisticRegression(max_iter=1000), scale=True)
pipe.fit(ds.X_train, ds.y_train)
names = output_feature_names(pipe)
print(f'encoded width: {len(names)}')
names[:12]

encoded width: 61


['tenure',
 'MonthlyCharges',
 'TotalCharges',
 'SeniorCitizen',
 'n_addons',
 'addon_adoption_rate',
 'realized_arpu',
 'price_vs_history',
 'billing_discrepancy',
 'charges_per_addon',
 'household_size',
 'is_new_customer']